**Notes from Bety:**

- Be sure to experiment with large populations and niterations ($\mathcal{O}(1000)$) early, as equations from smaller experiments don't always reflect larger runs with the same hyperparameters/settings (new "regimes" sometimes emerge). This includes new variables appearing as important (in place of others earlier in the run) or complicated expressions with many variables replaced by more nonlinear functions of fewer variables.
- `exp()` or more nonlinear operators (higher powers) require more constraints, and often nested constraints (i.e. `'exp':{'exp':0}`). In some cases, multiple nested (i.e. `'exp':{'exp':{'exp':0} ...}`) functions like $x_1^{x_2}$ and $x_1*exp(x_2^{x_3}$) can be dealt with by adjusting the `nested_constraints` (note this is more precise than using the `complexity_of_operators` parameter, which tends to get rid of certain operators altogether). Adjusting `complexity_of_variables` parameter can also help address these issues (especially when the power operator relies on multiple variables). 
- Arguments in the power operator tend to become complicated, the constraint `constraints={'^': (-1,1)}` helps limit the complexity of the power argument (you can experiment with more complex args `constraints={'^':(-1,2)}`). 
- The ratio `complexity_of_variables`/`complexity_of_constants` also is important (not just each individually) . Using `complexity_of_variables = 1` (or too small) leads to complicated combinations of variables early on often with the power operator, such as $x_1^{(x_2*x_3)}$.
- Note the best `complexity_of_variables` value will likely depend on number of variables (chosen for 8 input variables in this example).
- The native PySR `select_k_features` parameter didn't work well for me. For reducing number of variables in the final equations, increasing the complexity of variables and doing very large runs can help identify which variables are important.
- Batching is critical to make datasets of size $> ~\mathcal{O}(10^4)$ manageable. Compute is generally better spent on longer runs (with smaller batches) versus shorter runs with no batching (and more data points). 
- Refer to https://astroautomata.com/PySR/tuning/, although note some functionality mentioned is outdated or no longer supported
- Experiment with different variable scalings (e.g., z-scoring a strictly positive variable $x_1$ might get rid of any $x_1^a$ where $a<1$). In theory and depending on the complexity settings, the algorithm should recover $(x_1 + \mathrm{offset})^a$ but in practice it doesn't always and so scaling choices can change the form and performance of the equations.
- See https://ai.damtp.cam.ac.uk/pysr/api/ for the many options in PySRRegressor

In [ ]:
import os
os.environ['JULIA_DEPOT_PATH'] = '/pscratch/sd/s/sferrett/.julia'
os.environ['PYTHON_JULIAPKG_PROJECT'] = '/pscratch/sd/s/sferrett/.julia/environments/pyjuliapkg'
import gc
import json
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import xarray as xr
import pandas as pd
import proplot as pplt
from pysr import PySRRegressor, jl
jl.seval('safe_pow(x, y) = abs(x)^y')
pplt.rc.update({'figure.dpi':100})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)

SPLITSDIR    = CONFIGS['filepaths']['splits']
PREDSDIR     = CONFIGS['filepaths']['predictions']
WEIGHTSDIR   = CONFIGS['filepaths']['weights']
MODELSDIR    = CONFIGS['filepaths']['models']
TARGETVAR    = CONFIGS['variables']['target']
SRRUNS       = CONFIGS['experiments']['sr']['runs']
RUNNAME      = list(SRRUNS.keys())[0]
RUNCONFIG    = SRRUNS[RUNNAME]
FIELDVARS    = RUNCONFIG['fieldvars']
LOCALVARS    = RUNCONFIG['localvars']
FEATURESFROM = RUNCONFIG.get('features_from',None)
PREDICTORS   = FIELDVARS + LOCALVARS
OUTDIR       = os.path.join(MODELSDIR,'sr',RUNNAME)
SUBSETSIZE   = 5000
SEED         = 42
SEARCHMODE   = 'test'

print(f'Run: {RUNNAME}')
if FEATURESFROM:
    print(f'  Field vars (kernel-integrated from {FEATURESFROM} weights): {FIELDVARS}')
else:
    print(f'  Field vars (from normalized splits): {FIELDVARS}')
print(f'  Local vars (from normalized splits): {LOCALVARS}')
print(f'  Target: {TARGETVAR}')

In [ ]:
def kernel_integrate(fields,weights,dlev,mask=None):
    '''Apply learned kernel weights to vertical profiles (numpy).
    fields:  (nsamp, nfieldvars, nlevs)
    weights: (nfieldvars, nlevs)
    dlev:    (nlevs,)
    mask:    (nsamp, nlevs) or None
    Returns: (nsamp, nfieldvars)
    '''
    weighted = fields * weights[None,:,:] * dlev[None,None,:]
    if mask is not None:
        weighted = weighted * mask[:,None,:]
    return weighted.sum(axis=2)

def load_split(splitname,fieldvars,localvars,targetvar,splitsdir,
               weightsdir=None,featuresfrom=None,seedidx=0):
    splitpath = os.path.join(splitsdir,f'norm_{splitname}.h5')
    splitds   = xr.open_dataset(splitpath,engine='h5netcdf')
    refda = splitds[targetvar].transpose('time','lat','lon')
    ntime = splitds.sizes['time']
    columns = {}
    if featuresfrom:
        # Load learned kernel weights: dims (field, lev, seed)
        wpath = os.path.join(weightsdir,f'{featuresfrom}_{splitname}_weights.nc')
        wds   = xr.open_dataset(wpath,engine='h5netcdf')
        weights = wds['k1'].isel(seed=seedidx).values  # (nfieldvars, nlevs)
        wds.close()
        # Load raw profile fields + dlev + surfmask from split
        nlevs = splitds.sizes['lev']
        dlev  = splitds['dlev'].values
        fieldarrays = []
        for var in fieldvars:
            da  = splitds[var].transpose('time','lat','lon','lev')
            arr = da.values.reshape(-1,nlevs)
            fieldarrays.append(arr)
        fields3d = np.stack(fieldarrays,axis=1)  # (nsamp, nfieldvars, nlevs)
        mask = splitds['surfmask'].transpose('time','lat','lon','lev').values.reshape(-1,nlevs)
        feats = kernel_integrate(fields3d,weights,dlev,mask)  # (nsamp, nfieldvars)
        for i,var in enumerate(fieldvars):
            columns[var] = feats[:,i]
    else:
        for var in fieldvars:
            da = splitds[var]
            if 'time' in da.dims:
                arr = da.transpose('time','lat','lon').values
            else:
                arr = np.tile(da.values,(ntime,1,1))
            columns[var] = arr.ravel()
    for var in localvars:
        da = splitds[var]
        if 'time' in da.dims:
            arr = da.transpose('time','lat','lon').values
        else:
            arr = np.tile(da.values,(ntime,1,1))
        columns[var] = arr.ravel()
    X = pd.DataFrame(columns)
    y = refda.values.ravel()
    splitds.close()
    return X,y,refda

Xtrain,ytrain,refdatrain = load_split('train',FIELDVARS,LOCALVARS,TARGETVAR,SPLITSDIR,WEIGHTSDIR,FEATURESFROM)
Xvalid,yvalid,refdavalid = load_split('valid',FIELDVARS,LOCALVARS,TARGETVAR,SPLITSDIR,WEIGHTSDIR,FEATURESFROM)
print(f'Train: X={Xtrain.shape}, y={ytrain.shape}')
print(f'Valid: X={Xvalid.shape}, y={yvalid.shape}')

In [ ]:
def clean(X,y):
    df = X.copy()
    df['__target__'] = y
    df = df.replace([np.inf,-np.inf],np.nan).dropna()
    yclean = df.pop('__target__').values
    return df,yclean

Xtrain,ytrain = clean(Xtrain,ytrain)
Xvalid,yvalid = clean(Xvalid,yvalid)
print(f'Train after cleaning: X={Xtrain.shape}, y={ytrain.shape}')
print(f'Valid after cleaning: X={Xvalid.shape}, y={yvalid.shape}')
print(f'\nTrain X summary:\n{Xtrain.describe()}')
print(f'\nTrain y: mean={ytrain.mean():.6f}, std={ytrain.std():.6f}, min={ytrain.min():.6f}, max={ytrain.max():.6f}')

In [ ]:
np.random.seed(SEED)
if len(ytrain) > SUBSETSIZE:
    subidx = np.random.choice(len(ytrain),size=SUBSETSIZE,replace=False)
    Xsub = Xtrain.iloc[subidx].reset_index(drop=True)
    ysub = ytrain[subidx]
    print(f'Subsampled {SUBSETSIZE} from {len(ytrain)} training points')
else:
    Xsub = Xtrain.reset_index(drop=True)
    ysub = ytrain
    print(f'Using all {len(ytrain)} training points (< {SUBSETSIZE})')
print(f'Xsub: {Xsub.shape}, ysub: {ysub.shape}')

In [ ]:
nvars = len(PREDICTORS)
fig,axes = pplt.subplots(ncols=nvars,refwidth=3.5,refheight=3)
for i,var in enumerate(PREDICTORS):
    ax = axes[i]
    ax.scatter(Xsub[var],ysub,s=1,alpha=0.3)
    ax.format(xlabel=var,ylabel=TARGETVAR if i==0 else '')
fig.suptitle(f'{TARGETVAR} vs Predictors (n={len(ysub)})')
pplt.show()

In [ ]:
opcomplexity = {'+':1,'-':1,'*':1,'/':3,'safe_pow':3,'exp':4,'log':4}

searchparams = {
    'test':dict(niterations=5,populations=5,population_size=33,maxsize=15,timeout_in_seconds=300),
    'medium':dict(niterations=100,populations=20,population_size=50,maxsize=30,timeout_in_seconds=1800),
    'full':dict(niterations=10_000_000,populations=30,population_size=100,maxsize=50,timeout_in_seconds=int(60*60*7.5)),
}

params = searchparams[SEARCHMODE]
print(f'Search mode: {SEARCHMODE}')
print(f'Parameters: {params}')

In [ ]:
import time
os.makedirs(OUTDIR,exist_ok=True)

model = PySRRegressor(
    niterations=params['niterations'],
    populations=params['populations'],
    population_size=params['population_size'],
    binary_operators=['+','-','*','/','safe_pow'],
    unary_operators=['exp','log'],
    complexity_of_operators=opcomplexity,
    complexity_of_variables=2,
    complexity_of_constants=1,
    maxsize=params['maxsize'],
    maxdepth=4,
    constraints={'safe_pow':(-1,1)},
    nested_constraints={
        'exp':{'exp':0,'log':0,'safe_pow':0},
        'safe_pow':{'safe_pow':0},
        'log':{'log':0,'exp':0}},
    extra_sympy_mappings={'safe_pow':lambda x,y:x**y},
    loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    batching=len(ysub)>2000,
    batch_size=min(2000,len(ysub)),
    random_state=SEED,
    deterministic=True,
    multithreading=False,
    procs=0,
    tempdir=OUTDIR,
    temp_equation_file=True,
    delete_tempfiles=False,
    timeout_in_seconds=params['timeout_in_seconds'],
    progress=True)

t0 = time.time()
model.fit(Xsub.values,ysub,variable_names=PREDICTORS)
elapsed = time.time()-t0

print(f'\nSearch completed in {elapsed/60:.1f} minutes')
print(model)

In [ ]:
equations = model.equations_
print(equations[['equation','complexity','loss','score']].to_string())

In [ ]:
besteq = model.get_best()
ypredtrain = model.predict(Xtrain.values)
ypredvalid = model.predict(Xvalid.values)
msetrain = np.mean((ytrain-ypredtrain)**2)
msevalid = np.mean((yvalid-ypredvalid)**2)

print(f'Best equation: {besteq["equation"]}')
print(f'  Complexity: {besteq["complexity"]}')
print(f'  MSE train:  {msetrain:.6f}')
print(f'  MSE valid:  {msevalid:.6f}')

In [ ]:
fig,ax = pplt.subplots(refwidth=5,refheight=3.5)
ax.scatter(equations['complexity'],equations['loss'],zorder=5)
ax.format(xlim=(0,equations['complexity'].max()+3),
          ylim=(equations['loss'].min()*0.9,equations['loss'].max()*1.1),
          xlabel='Complexity',ylabel='MSE Loss',
          title='Pareto Frontier of Discovered Equations')
for i,row in equations.iterrows():
    label = str(row['equation'])[:50]
    ax.annotate(label,xy=(row['complexity'],row['loss']),xytext=(5,5),textcoords='offset points',fontsize=5,clip_on=True)
fig.savefig(os.path.join(OUTDIR,'pareto_frontier.png'),dpi=150)
pplt.show()

In [ ]:
import pickle
from scripts.data.classes import PredictionWriter

modelpath = os.path.join(OUTDIR,f'{RUNNAME}.pkl')
with open(modelpath,'wb') as f:
    pickle.dump(model,f)

equations.to_csv(os.path.join(OUTDIR,'equations.csv'),index=False)

writer = PredictionWriter(SPLITSDIR,targetvar=TARGETVAR)
for splitname,ypred,refda in [('train',ypredtrain,refdatrain),('valid',ypredvalid,refdavalid)]:
    Xraw,yraw,_ = load_split(splitname,FIELDVARS,LOCALVARS,TARGETVAR,SPLITSDIR,WEIGHTSDIR,FEATURESFROM)
    valid   = np.isfinite(Xraw).all(axis=1).values & np.isfinite(yraw)
    predarr = writer.predictions_to_array(ypred,valid,refda)
    predds  = writer.predictions_to_dataset(predarr[...,np.newaxis],refda)
    writer.save(RUNNAME,predds,'predictions',splitname,PREDSDIR)

metadata = {
    'run':RUNNAME,'features_from':FEATURESFROM,'predictors':PREDICTORS,
    'target':TARGETVAR,'subset_size':len(ysub),'train_size':len(ytrain),
    'valid_size':len(yvalid),'search_mode':SEARCHMODE,'search_params':params,
    'mse_train':float(msetrain),'mse_valid':float(msevalid),
    'best_equation':str(besteq['equation']),'elapsed_minutes':elapsed/60,'seed':SEED}
with open(os.path.join(OUTDIR,'metadata.json'),'w') as f:
    json.dump(metadata,f,indent=2)

print(f'Model saved to: {modelpath}')
print(f'Predictions saved to: {PREDSDIR}')
print(f'Best equation: {besteq["equation"]}')
gc.collect()

In [2]:
# FILEDIR = '/global/cfs/cdirs/m4334/sferrett/monsoon-discovery/data/interim'
# x  = xr.open_dataarray(f'{FILEDIR}/cape.nc').isel(time=slice(0,5),lat=slice(0,5),lon=slice(0,5)).values.ravel()
# y  = xr.open_dataarray(f'{FILEDIR}/bl.nc').isel(time=slice(0,5),lat=slice(0,5),lon=slice(0,5)).values.ravel()
# df = pd.DataFrame({'x':x,'y':y})

# df = df.replace([np.inf,-np.inf],np.nan).dropna()
# print(f'Data shape after cleaning: {df.shape}')
# print(df.describe())

# fig,ax = pplt.subplots(nrows=1,ncols=1)
# ax.format(xlabel='$CAPE_L$ (K)',ylabel='$B_L$ (m/s$^2$)')
# ax.scatter(df['x'],df['y'],alpha=0.6)
# pplt.show()

# model = PySRRegressor(
#     niterations=3,
#     populations=3,
#     population_size=33,
#     tournament_selection_n=2,
#     binary_operators=['+','-','*','/','safe_pow'],
#     unary_operators=['exp','log'],
#     complexity_of_operators={'+':1,'*':1,'-':1,'/':3,'safe_pow':3,'exp':4,'log':4},
#     complexity_of_variables=2,
#     complexity_of_constants=1,
#     maxsize=10,
#     constraints={'safe_pow':(-1,1)},
#     nested_constraints={
#         'exp':{'exp':0,'log':0,'safe_pow':0},
#         'safe_pow':{'safe_pow':0},
#         'log':{'log': 0,'exp':0}},  
#     extra_sympy_mappings={'safe_pow':lambda x,y:x**y},
#     batching=False,
#     batch_size=100,
#     loss='loss(x, y) = (x - y)^2',
#     model_selection='best',
#     random_state=42,
#     deterministic=True,
#     multithreading=False,
#     procs=0,
#     timeout_in_seconds=300)

# model.fit(df[['x']].values, df['y'].values)
# print(model)

# equations = model.equations_
# print(equations[['equation','complexity','loss','score']].head(10))

# best  = model.get_best()
# eqstr = best['sympy_format']
# ypred = best['lambda_format'](df[['x']].values)

# fig,ax = pplt.subplots(nrows=1,ncols=1,refheight=2,refwidth=2)
# ax.format(title='Pareto Frontier of Discovered Equations',xlabel='Complexity',ylabel='MSE Loss')
# ax.scatter(equations['complexity'],equations['loss'])
# for i,row in equations.iterrows():
#     ax.annotate(row['equation'],xy=(row['complexity'],row['loss']),xytext=(5,5),textcoords='offset points',fontsize=6)
# pplt.show()

# fig,ax = pplt.subplots(nrows=1,ncols=1,refwidth=2.5)
# ax.format(title=f'Best PySR Equation: {eqstr}',grid=True,xlabel='$y_{true}$',ylabel='$y_{pred}$')
# ax.plot(df['y'],df['y'],color='k',linestyle='--')
# ax.scatter(df['y'],ypred,alpha=0.6)
# pplt.show()